# Automotive market: exploration

Rough work before writing `src/`. Checks: seasonality, the 2020 lockdown dip, and
whether EV adoption really looks like an S-curve (logistic) before I build the forecast on it.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data import load_all
from src import market as mk, forecast as fc

market, models, reviews = load_all()
market.head()

## 1. Total market by month: seasonality and the 2020 dip

In [ ]:
monthly = market.groupby("month")["units"].sum()
ax = (monthly / 1e3).plot(figsize=(9, 3.5), color="#2f5d8a")
ax.set_ylabel("units (thousands)")
plt.show()

# average by calendar month, relative to the yearly mean
rel = monthly / monthly.groupby(monthly.index.year).transform("mean")
rel.groupby(rel.index.month).mean().round(2)

October-November is ~15-20% above average (festive season) and April-June 2020 collapses
(lockdown). So: yearly totals for trends, and 2020 excluded when fitting market growth.

## 2. Fuel mix

In [ ]:
units, shares = mk.yearly_fuel_mix(market)
shares

## 3. Is EV adoption an S-curve?
If share = L / (1 + exp(-k(t - t0))), then log(share / (L - share)) is a straight line in t.
Early in the curve (share much smaller than L) it is almost the same as log(share), i.e. exponential.

In [ ]:
ev = fc.monthly_ev_share(market)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(ev.t, 100 * ev.ev_share)
axes[0].set_title("EV share (%)")
for L in [0.3, 0.5, 0.7]:
    axes[1].plot(ev.t, np.log(ev.ev_share / (L - ev.ev_share)), label=f"L={L}")
axes[1].set_title("logit transform")
axes[1].legend()
plt.show()

All three are close to straight lines, so any of these saturation levels fits the past.
The data can't tell me L. I'll use it as a scenario input instead of fitting it.

In [ ]:
fc.backtest(ev)

## 4. Company X in the EV segment

In [ ]:
mk.ev_segment_shares(market)

In [ ]:
mk.ev_model_mix(models)

In [ ]:
mk.review_summary(reviews)

X-City EV: lowest range and charging scores, and its share of company EV sales keeps falling.
That's the main input for the recommendations.